In [2]:
from diffusers import (
    DiffusionPipeline, 
    StableDiffusionPipeline,
    UNet2DConditionModel,
    PNDMScheduler,
    AutoencoderKL,
    LMSDiscreteScheduler
)
from diffusers import AutoencoderKL
from transformers import CLIPProcessor, CLIPModel, CLIPTextModel, CLIPTokenizer

from PIL import Image

import cv2
import torch
import numpy as np

## Load Stable Diffusion Model 

### Hyperparameters

In [6]:
PROMPT = ["A dog riding a motorbike"]
num_samples = 10
num_inference_steps = 50

### Stable Diffusion Pipeline

In [7]:
# diffusion_pipeline = DiffusionPipeline.from_pretrained("stable-diffusion-v1-5/stable-diffusion-v1-5", use_safetensors=True)
# diffusion_pipeline.to("cuda")
# images = diffusion_pipeline(PROMPT,
#                             num_inference_steps = num_inference_steps,
#                             num_images_per_prompt = num_samples
#         ).images

### Loading Stable Diffusion Model

In [8]:
# Load individual model components
model_id = "runwayml/stable-diffusion-v1-5"

# Load VAE for decoding latents
vae = AutoencoderKL.from_pretrained(model_id, subfolder="vae").cuda()

# Load Text Encoder and Tokenizer
tokenizer = CLIPTokenizer.from_pretrained(model_id, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(model_id, subfolder="text_encoder").cuda()

# Load UNet (the diffusion model)
unet = UNet2DConditionModel.from_pretrained(model_id, subfolder="unet").cuda()

# Load scheduler (LMS or PNDM recommended for inference)
scheduler = LMSDiscreteScheduler.from_pretrained(model_id, subfolder="scheduler")

### Generating Images

In [9]:
# Function to encode text prompt into embeddings
def get_text_embeddings(prompt, batch_size):
    text_input = tokenizer([str(prompt)] * batch_size, padding="max_length", max_length=77, return_tensors="pt")
    with torch.no_grad():
        text_embeddings = text_encoder(text_input.input_ids.cuda())[0]
    
    # Generate unconditional embeddings for classifier-free guidance
    uncond_input = tokenizer([""] * batch_size, padding="max_length", max_length=77, return_tensors="pt")
    with torch.no_grad():
        uncond_embeddings = text_encoder(uncond_input.input_ids.cuda())[0]
    
    # Concatenate conditional & unconditional embeddings
    text_embeddings = torch.cat([uncond_embeddings, text_embeddings])  # Shape: (2 * batch_size, 77, 768)
    return text_embeddings


def generate_images_from_latents(latents, prompt, num_inference_steps=50, guidance_scale=7.5):
    batch_size = latents.shape[0]  # Number of images in batch
    text_embeddings = get_text_embeddings(prompt, batch_size)

    scheduler.set_timesteps(num_inference_steps)
    latents = latents * scheduler.init_noise_sigma  # Scale initial noise

    for t in scheduler.timesteps:
        # Expand latents for classifier-free guidance (double batch size)
        latent_model_input = torch.cat([latents] * 2)
        latent_model_input = scheduler.scale_model_input(latent_model_input, timestep=t)

        # Predict noise
        with torch.no_grad():
            noise_pred = unet(latent_model_input, t, encoder_hidden_states=text_embeddings).sample

        # Apply classifier-free guidance
        noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
        noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)

        # Denoising step
        latents = scheduler.step(noise_pred, t, latents).prev_sample

    # Decode latents into images
    with torch.no_grad():
        images = vae.decode(latents / 0.18215).sample  # Scale down before decoding

    # Convert tensor to PIL images
    images = (images / 2 + 0.5).clamp(0, 1)  # Normalize to [0,1]
    images = (images * 255).cpu().numpy().astype(np.uint8)  # Convert to uint8
    pil_images = [Image.fromarray(img.transpose(1, 2, 0)) for img in images]  # CHW → HWC

    return pil_images

In [6]:
# latent_vector = torch.randn((num_samples, 4, 64, 64)).cuda()  # Custom latent
# generated_images = generate_images_from_latents(latent_vector, PROMPT)

In [7]:
# generated_images[0]

## Set up Scoring Functions

In [10]:
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [9]:
# Obtain the text-image similarity scores
# with torch.no_grad():
#     inputs = clip_processor(text=PROMPT, 
#                     images=generated_images, return_tensors="pt", padding=True)
#     outputs = clip_model(**inputs)

In [10]:
# logits_per_image = outputs.logits_per_image # this is the image-text similarity score

In [11]:
# logits_per_image

In [12]:
# def average_hue(image):
#     hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
#     hue, sat, val = cv2.split(hsv)
#     hue = hue[sat > 50]  # Ignore low-saturation (grayscale-like) areas
#     return np.mean(hue) if hue.size > 0 else 0

# avg_hue_score = average_hue(generated_images[5])
# print(f"Average Hue: {avg_hue_score}")

In [13]:
# def brightness(image):
#     gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
#     brightness_score = np.mean(gray)
#     return brightness_score

# brightness_score = brightness(generated_images[3])
# print(f"Brightness: {brightness_score}")

## Set up MAP-Elites

In [3]:
import os

from IPython.display import clear_output
import functools
import time

import jax
import jax.numpy as jnp

from qdax.core.map_elites import MAPElites
from qdax.core.emitters.mutation_operators import isoline_variation
from qdax.core.containers.mapelites_repertoire import compute_cvt_centroids
from qdax.core.emitters.standard_emitters import MixingEmitter
from qdax.utils.plotting import plot_map_elites_results

from qdax.utils.metrics import CSVLogger, default_qd_metrics

from jax.flatten_util import ravel_pytree

from functools import partial

if "COLAB_TPU_ADDR" in os.environ:
  from jax.tools import colab_tpu
  colab_tpu.setup_tpu()

clear_output()

### MAP-Elites Hyperparameters

In [4]:
batch_size = 4
num_iterations = 100 
seed = 42 
iso_sigma = 0.005 
line_sigma = 0.05 
num_init_cvt_samples = 50000 
num_centroids = 1024 
min_descriptor = 0. 
max_descriptor = 255.0 

### Set up Scoring Function and Metrics

In [11]:
def brightness(image):
    """Compute brightness by averaging pixel values."""
    return jnp.mean(image)  # No OpenCV needed!

def average_hue(image):
    """Compute an approximate average hue using JAX operations."""
    r, g, b = image[..., 0], image[..., 1], image[..., 2]
    max_c = jnp.max(image, axis=-1)
    min_c = jnp.min(image, axis=-1)
    delta = max_c - min_c

    hue = jnp.where(
        delta == 0, 0,
        jnp.where(max_c == r, (60 * ((g - b) / delta) + 360) % 360,
        jnp.where(max_c == g, (60 * ((b - r) / delta) + 120) % 360,
                             (60 * ((r - g) / delta) + 240) % 360
        )))
    
    return jnp.mean(hue)  # Compute mean hue across image


def scoring_function(genotypes,
        key,
        prompt,
        clip_model,
        clip_processor,
):
    # Convert jax genotypes to pytorch
    latents = torch.tensor(np.array(genotypes), dtype=torch.float32).cuda()    
    
    # Generate images
    generated_images = generate_images_from_latents(latents, prompt)
    
    generated_images_np = np.array([np.array(img) for img in generated_images])  
    generated_images_jnp = jnp.array(generated_images_np, dtype=jnp.float32)  # Ensure JAX dtype
    
    # Generate CLIP Scores
    inputs = clip_processor(text=prompt, 
                    images=generated_images, return_tensors="pt", padding=True)
    outputs = clip_model(**inputs)
    
    # Convert fitnesses to jax arrays
    fitnesses_torch = outputs.logits_per_image  # PyTorch tensor
    fitnesses_np = fitnesses_torch.detach().cpu().numpy()  # Convert to NumPy
    fitnesses = jnp.array(fitnesses_np, dtype=jnp.float32)  # Convert to JAX dtype
    
    # Calculate descriptors    
    descriptor_1 = jax.vmap(brightness)(generated_images_jnp)
    descriptor_2 = jax.vmap(average_hue)(generated_images_jnp)
    descriptors = jnp.stack([descriptor_1, descriptor_2], axis=-1)  # Shape: (batch_size, 2)    

    return fitnesses, descriptors, {}

scoring_fn = partial(
    scoring_function,
    prompt=PROMPT,
    clip_model=clip_model,
    clip_processor=clip_processor
    )

In [12]:
# Define a metrics function
metrics_function = functools.partial(
    default_qd_metrics,
    qd_offset = 0.0
)

### Initialise Genotypes and Emitter

In [5]:
key = jax.random.key(seed)

key, subkey = jax.random.split(key)  # JAX PRNG handling
init_genotypes = jax.random.normal(subkey, shape=(batch_size, 4, 64, 64), dtype=jnp.float32)

In [13]:
# Define emitter
variation_fn = functools.partial(
    isoline_variation, iso_sigma=iso_sigma, line_sigma=line_sigma
)

mixing_emitter = MixingEmitter(
    mutation_fn=None,
    variation_fn=variation_fn,
    variation_percentage=1.0,
    batch_size=batch_size
)

### Set up MAP-Elites


In [14]:
# Instantiate MAP-Elites
map_elites = MAPElites(
    scoring_function=None,
    emitter=mixing_emitter,
    metrics_function=metrics_function,
)

# Compute the centroids
key, subkey = jax.random.split(key)
centroids = compute_cvt_centroids(
    num_descriptors=2,
    num_init_cvt_samples=num_init_cvt_samples,
    num_centroids=num_centroids,
    minval=min_descriptor,
    maxval=max_descriptor,
    key=subkey,
)

In [15]:
# Evaluate the initial population
key, subkey = jax.random.split(key)
fitnesses, descriptors, extra_scores = scoring_fn(init_genotypes, subkey)

In [16]:
# Compute initial repertoire and emitter state
repertoire, emitter_state, metrics = map_elites.init_ask_tell(
    genotypes=init_genotypes,
    fitnesses=fitnesses,
    descriptors=descriptors,
    centroids=centroids,
    key=key,
    extra_scores=extra_scores,
)

In [17]:
def generate_image_from_genotype(image_array, prompt):
    """Displays an image from a NumPy array."""
    # Convert jax genotypes to pytorch
    latents = torch.tensor(np.array(image_array), dtype=torch.float32).cuda()    
    
    # Generate images
    generated_images = generate_images_from_latents(latents, prompt)

    return generated_images

In [18]:
init_generated_images = generate_image_from_genotype(init_genotypes, PROMPT)

In [ ]:
from APDDv2.demo import load_model, get_score
import APDDv2.models.clip as clip


def creativity_score_fn(image, preprocess_model, score_model):
    print("Loading image")
    image =  Image.fromarray(np.array(image).astype('uint8'), 'RGB')
    print("Preprocessing image")
    image_input = preprocess(image).unsqueeze(0).to(device)
    print("Predicting creativity score")
    creativity_score = creativity_model(image_input)
    
    _, creativity_score = get_score(None, creativity_score)
    return creativity_score

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading preprocess")
_, preprocess = clip.load('ViT-B/16', device)
print("Loading creativity model")
creativity_model = load_model("APDDv2/model_weights/3.Creativity_reg_weight--e5-train0.4212-test0.7122_best.pth", device)


creativity_score_fn(init_generated_images[0], preprocess, creativity_model)

Loading preprocess
Loading creativity model
Loading image
Preprocessing image
Predicting creativity score


array(5.2253485, dtype=float32)

In [ ]:
def save_images(images, filenames="", folder="results"):
    """Saves a batch of images into a results folder."""
    
    # Ensure the directory exists
    os.makedirs(folder, exist_ok=True)
    
    if filenames=="":
        filenames = [f"image_sample_{i+1}.png" for i in range(len(images))]
    
    print("filenames:", filenames)

    for i, image_array in enumerate(images):
        # Convert to NumPy if needed
        image_array = np.array(image_array)

        # Normalize to [0,255] if needed
        image_array = (image_array - image_array.min()) / (image_array.max() - image_array.min() + 1e-8)
        image_array = (image_array * 255).astype(np.uint8)

        # Handle shape issues
        if image_array.shape == (512, 512, 3):  # Already in correct format
            image = Image.fromarray(image_array, mode="RGB")
        elif image_array.shape[0] == 3:  # If (3, H, W) -> Convert to (H, W, 3)
            image_array = np.transpose(image_array, (1, 2, 0))
            image = Image.fromarray(image_array, mode="RGB")
        elif image_array.shape[0] == 1:  # If (1, H, W) -> Convert to (H, W)
            image_array = np.squeeze(image_array, axis=0)
            image = Image.fromarray(image_array, mode="L")
        elif image_array.ndim == 2:  # If already (H, W), use grayscale
            image = Image.fromarray(image_array, mode="L")
        else:
            raise ValueError(f"Unexpected shape {image_array.shape} for image {i}")

        # Save image
        image.save(os.path.join(folder, filenames[i]))

# Example usage:
save_images(init_generated_images)  # Replace with your batch of images

### Run MAP-Elites

In [ ]:
metrics = dict.fromkeys(["iteration", "qd_score", "coverage", "max_fitness", "time"], jnp.array([]))
csv_logger = CSVLogger(
    "results/mapelites-logs.csv",
    header=list(metrics.keys())
)

# Add a progress bar if tqdm is installed
try:
    from tqdm import tqdm
    bar = tqdm(range(num_iterations))
except ImportError:
    bar = range(num_iterations)

# Main loop
for i in bar:
    start_time = time.time()
    key, subkey = jax.random.split(key)
    # Generate solutions
    genotypes, extra_info = map_elites.ask(repertoire, emitter_state, subkey)

    # Evaluate solutions: get fitness, descriptor and extra scores.
    # This is where custom evaluations on CPU or GPU can be added.
    key, subkey = jax.random.split(key)
    fitnesses, descriptors, extra_scores = scoring_fn(genotypes, subkey)

    # Update MAP-Elites
    repertoire, emitter_state, current_metrics = map_elites.tell(
        genotypes=genotypes,
        fitnesses=fitnesses,
        descriptors=descriptors,
        repertoire=repertoire,
        emitter_state=emitter_state,
        extra_scores=extra_scores,
        extra_info=extra_info,
    )

    timelapse = time.time() - start_time

    # Metrics
    current_metrics["iteration"] = i
    current_metrics["time"] =timelapse
    current_metrics = jax.tree.map(lambda x: jnp.array([x]), current_metrics)
    metrics = jax.tree.map(lambda metric, current_metric: jnp.concatenate([metric, current_metric], axis=0), metrics, current_metrics)

    # Log
    csv_logger.log(jax.tree.map(lambda x: x[-1], metrics))

### Save Repertoire

In [ ]:
import numpy as np

def save_repertoire(filename, genotypes, descriptors, fitnesses):
    """Saves the repertoire (genotypes, descriptors, fitnesses) as a .npz file."""
    np.savez(filename, genotypes=genotypes, descriptors=descriptors, fitnesses=fitnesses)
    print(f"Repertoire saved to {filename}")

# Example usage
save_repertoire("results/repertoire.npz", init_genotypes, descriptors, fitnesses)

### Visualisation

In [ ]:

# Create the x-axis array
env_steps = metrics["iteration"]

%matplotlib inline
# Create the plots and the grid
fig, axes = plot_map_elites_results(env_steps=env_steps, metrics=metrics, repertoire=repertoire, min_descriptor=min_descriptor, max_descriptor=max_descriptor)

In [ ]:
import numpy as np
import os
import random

def save_final_images(repertoire, save_images_fn, prompt, n_sample=5, folder="results"):
    """
    Saves non-empty solutions from the repertoire using the provided save_images function.
    Samples the highest fitness genotype and n_sample other genotypes.
    
    Parameters:
    - repertoire: Dictionary containing 'genotypes', 'descriptors', and 'fitnesses'.
    - save_images_fn: Function to save images (expects (images, filenames, folder) as input).
    - prompt: The prompt used for generating the images from genotypes.
    - n_sample: The number of additional genotypes to sample.
    - folder: Directory to save the final images.
    """
    os.makedirs(folder, exist_ok=True)  # Ensure the directory exists

    valid_genotypes = []
    fitnesses = []
    descriptors = []
    filenames = []

    # Collect valid genotypes, descriptors, fitnesses, and filenames
    for i, (fitness, descriptor, genotype) in enumerate(zip(repertoire.fitnesses, repertoire.descriptors, repertoire.genotypes)):
        if fitness == -jnp.inf:  # Skip empty solutions
            continue
        valid_genotypes.append(genotype)
        fitnesses.append(fitness[0])
        descriptors.append(descriptor)
        filename = f"final_image_{fitness[0]}_{descriptor[0]:.4f}_{descriptor[1]:.4f}.png"
        filenames.append(filename)

    # Step 1: Find the genotype with the highest fitness
    best_fitness_idx = np.argmax(fitnesses)
    best_genotype = valid_genotypes[best_fitness_idx]
    best_filename = filenames[best_fitness_idx]

    # Step 2: Sample `n_sample` other valid genotypes (and filenames)
    # Remove the best genotype from the valid genotypes list to avoid selecting it again
    remaining_genotypes = valid_genotypes[:best_fitness_idx] + valid_genotypes[best_fitness_idx+1:]
    remaining_filenames = filenames[:best_fitness_idx] + filenames[best_fitness_idx+1:]
    
    # Sample `n_sample` from the remaining genotypes and filenames
    sampled_genotypes_indices = random.sample(range(len(remaining_genotypes)), n_sample)
    sampled_genotypes = [remaining_genotypes[i] for i in sampled_genotypes_indices]
    sampled_filenames = [remaining_filenames[i] for i in sampled_genotypes_indices]

    # Combine the best genotype with the sampled ones
    combined_genotypes = [best_genotype] + sampled_genotypes
    combined_filenames = [best_filename] + sampled_filenames
    
    # Step 3: Generate images from combined genotypes
    final_images = generate_image_from_genotype(combined_genotypes, prompt)
    
    # Use the existing save_images function to save the images
    save_images_fn(final_images, combined_filenames, folder)

save_final_images(repertoire, save_images, PROMPT, n_sample=4)